<a href="https://colab.research.google.com/github/LNDOTIS/FlyRank-AI-Internship---Machine-Learning/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

```text
            PREDICTION / DECISION MOMENT
                              │
                              ▼
     Feature Window           │        Target Window
     Previous 30 Days         │        Next 30 Days
──────────────────────────────┼──────────────────────────
      Historical Data         │        Future Outcome
        Known Today           │        Not Known Today


                              │
        FEATURES              │           LABEL

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

For this lane, one row represents **one content page for one client at one prediction point**.

The page is identified by the combination of:

- `client_hash_id`
- `content_hash_id`

The model uses historical information available before the prediction moment to rank pages by their future decline risk.


---



### Feature window

For this notebook, the feature window is a 30-day period in **March 2026**.

The features are calculated from observations available during this historical window.

Conceptually:

```Previous 30 days → Prediction point → Future 30 days```

Historical feature window:
```2026-03-01``` to ```2026-03-31```

Future target window:
The following 30-day period after the prediction point.

The preferred future label is:

`is_declining = 1` if ```the page's impressions decrease by more than 20% in the future target window compared with the preceding feature window.```

The future target window is not used to construct features.

This separation is important because the goal is to simulate a real prediction made before the future outcome is known.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The five features selected for the first feature frame are:

1. `imp_prev30`
   - Total Google Search Console impressions during the previous 30 days.
   - Available at the prediction moment because this period has already occurred.

2. `clk_prev30`
   - Total Google Search Console clicks during the previous 30 days.
   - Available at the prediction moment because these clicks were observed before the prediction point.

3. `avg_position_prev30`
   - Average Google Search Console ranking position during the previous 30 days.
   - Available at the prediction moment because the ranking observations belong to the historical feature window.

4. `visible_queries`
   - Number of visible queries associated with the page in the available 90-day query window.
   - This describes the page's historical search visibility and can be used as a feature if the query window ends at or before the prediction point.

5. `content_age_days`
   - Age of the content at the prediction point.
   - Available at the prediction moment because the content creation date is known.


---


### Label

The preferred label is:

`is_declining`

A page receives:

- `1` if impressions decline by more than 20% in the future target window compared with the preceding feature window.
- `0` otherwise.

This is an observed future outcome, not a manually defined current status.

The label is calculated only after the future window has occurred and must never be used as a model feature.


---


### Context

The following fields are used for identification, grouping, joining, validation, or interpretation but are not model features:

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `keyword_hash_id`
- `url_hash_id`

`client_hash_id` is particularly important for client-grouped validation because pages from the same client may share patterns.


---


### Excluded

The following information is deliberately excluded from the feature set:

- Future-window impressions
- Future-window clicks
- Future-window ranking position
- `trend_direction` when it is calculated from the same or future period as the target
- `trend_pct` when it is calculated using information from the target window
- Any product decision fields such as `priority_score`, `health_score`, or `action_type`

These fields are excluded because they either contain future information or represent an existing product decision that the model should not simply learn to reproduce.

The target label itself is also excluded from the feature matrix.

The central rule is:

> Features must be knowable at the prediction moment. The future outcome is used only to construct the label.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


### Query 1: Verify the grain

The intended grain of `fact_content_daily_performance` is:

> One row per client, content page, and report date.

The following query checks whether the combination of
`report_date`, `client_hash_id`, and `content_hash_id`
appears more than once.

In [11]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(1) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(1) > 1
""")


In [12]:
grain_check.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Since this query returns zero rows, there are no duplicate combinations in the March 2026 slice, supporting the intended grain:

```one client × one content page × one day```.

### Query 2: Verify the March 2026 slice

The March 2026 slice is used as a mid-panel development period.

This query checks the number of daily fact rows and the minimum and maximum dates available in the slice.

In [13]:
march_stats = con.sql(f"""
    SELECT
        COUNT(1) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [14]:
march_stats

,row_count,clients,content_pages,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


The result shows the size of the March 2026 daily panel and confirms the date range used for the feature-development slice.

The number of rows is expected to be larger than the number of content pages because the table has daily grain: a page can contribute multiple observations across different dates.

### Query 3: Verify Google Search Console availability

The decline target is based on organic search impressions, so GSC availability is required.

The query compares the March 2026 slice before and after filtering with:

`gsc_data_available IS TRUE`

This verifies how many rows have observable GSC data rather than treating missing search tracking as zero performance.

In [15]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) * 100.0 / COUNT(*) AS gsc_available_pct
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
availability_check

,total_rows,gsc_available_rows,gsc_available_pct
0,9841378,3611061,36.692636


This query shows how many March 2026 daily observations have GSC data available.

The `IS TRUE` condition is important because a missing or false availability flag should not be interpreted as zero impressions. A page may have no recorded search data because GSC tracking was not yet available for that client.

## Five Initial Features

The first feature frame contains at most five features.

| Feature | Description | Available when? |
|---|---|---|
| `imp_prev30` | Total GSC impressions in the previous 30 days | Knowable at the prediction moment because the previous 30-day window has already ended |
| `clk_prev30` | Total GSC clicks in the previous 30 days | Knowable at the prediction moment because clicks were observed before the prediction moment |
| `avg_position_prev30` | Average GSC position in the previous 30 days | Knowable at the prediction moment because ranking observations come from the historical window |
| `visible_queries` | Number of visible queries associated with the page | Knowable at the prediction moment only if the 90-day query window ends on or before the prediction moment |
| `content_age_days` | Number of days since content creation | Knowable at the prediction moment because the content creation date is already known |

These features represent different aspects of a page:

- historical search visibility,
- historical search engagement,
- ranking performance,
- search query breadth,
- and content lifecycle.

The feature set is intentionally small for the first iteration. Additional features can be evaluated later after leakage checks.

## The leakage trap

> add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it.



In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score
)

In [38]:

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-04-01"

TARGET_START = "2026-04-01"
TARGET_END = "2026-05-01"

In [39]:
features = con.sql(f"""
    WITH feature_window AS (

        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS imp_prev30,

            SUM(gsc_clicks) AS clk_prev30,

            AVG(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN gsc_avg_position
                END
            ) AS avg_position_prev30,

            COUNT(
                CASE
                    WHEN gsc_impressions > 0
                    THEN 1
                END
            ) AS days_with_impressions

        FROM {TABLES["fact_daily"]}

        WHERE report_date >= DATE '{FEATURE_START}'
          AND report_date < DATE '{FEATURE_END}'

          -- Only use rows where GSC data actually exists
          AND gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id

    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,

        f.imp_prev30,
        f.clk_prev30,
        f.avg_position_prev30,
        f.days_with_impressions,

        -- Content age at prediction moment
        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '{TARGET_START}'
        ) AS content_age_days

    FROM feature_window f

    LEFT JOIN {TABLES["dim_content"]} c

        ON f.client_hash_id = c.client_hash_id
        AND f.content_hash_id = c.content_hash_id

    -- Require meaningful historical search volume
    WHERE f.imp_prev30 >= 100
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [40]:
print(f"Number of pages: {len(features):,}")

features.head()

Number of pages: 101,441


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,days_with_impressions,content_age_days
0,client_2094c6eb080311d5,content_14c17f59aa610ab3,122.0,2.0,5.882498,8,43
1,client_2094c6eb080311d5,content_15565677b6e1792f,148.0,1.0,12.350803,16,21
2,client_2094c6eb080311d5,content_15770c63daac443b,2585.0,4.0,6.251382,23,56
3,client_2094c6eb080311d5,content_157db5a38382c639,1143.0,4.0,6.104022,20,56
4,client_2094c6eb080311d5,content_15f758d8bd5341c5,347.0,0.0,46.934948,31,43


In [41]:
future_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_next30,

        SUM(gsc_clicks) AS clk_next30

    FROM {TABLES["fact_daily"]}

    WHERE report_date >= DATE '{TARGET_START}'
      AND report_date < DATE '{TARGET_END}'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [42]:
future_outcomes.head()

,client_hash_id,content_hash_id,imp_next30,clk_next30
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,249.0,1.0
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,281.0,0.0
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,722.0,0.0
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,3068.0,1.0
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,3896.0,3.0


In [45]:
data = features.merge(
    future_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)
data.head()

,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,days_with_impressions,content_age_days,imp_next30,clk_next30
0,client_2094c6eb080311d5,content_14c17f59aa610ab3,122.0,2.0,5.882498,8,43,377.0,0.0
1,client_2094c6eb080311d5,content_15565677b6e1792f,148.0,1.0,12.350803,16,21,59.0,0.0
2,client_2094c6eb080311d5,content_15770c63daac443b,2585.0,4.0,6.251382,23,56,10112.0,12.0
3,client_2094c6eb080311d5,content_157db5a38382c639,1143.0,4.0,6.104022,20,56,1892.0,7.0
4,client_2094c6eb080311d5,content_15f758d8bd5341c5,347.0,0.0,46.934948,31,43,167.0,0.0


In [46]:
data["impression_change_pct"] = (
    (data["imp_next30"] - data["imp_prev30"])
    / data["imp_prev30"]
)

In [47]:
data["is_declining"] = (
    data["impression_change_pct"] < -0.20
).astype(int)

In [48]:
print(
    data["is_declining"]
    .value_counts()
    .rename({0: "not_declining", 1: "declining"})
)

print(
    data["is_declining"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

is_declining
declining        51944
not_declining    48949
Name: count, dtype: int64
is_declining
1    51.48
0    48.52
Name: proportion, dtype: float64


In [49]:
feature_cols = [
    "imp_prev30",
    "clk_prev30",
    "avg_position_prev30",
    "days_with_impressions",
    "content_age_days"
]

In [50]:
model_data = data.dropna(
    subset=feature_cols + ["is_declining"]
).copy()

In [51]:
X = model_data[feature_cols]
y = model_data["is_declining"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100893, 5)
y shape: (100893,)


In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)
honest_prob = honest_model.predict_proba(X_test)[:, 1]

In [56]:
print("HONEST MODEL")
print("=" * 50)

print(
    classification_report(
        y_test,
        honest_pred,
        digits=3
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(y_test, honest_prob),
        3
    )
)

HONEST MODEL
              precision    recall  f1-score   support

           0      0.663     0.638     0.650     12238
           1      0.670     0.695     0.682     12986

    accuracy                          0.667     25224
   macro avg      0.667     0.666     0.666     25224
weighted avg      0.667     0.667     0.667     25224

ROC-AUC: 0.731


In [59]:
leaky_feature_cols = feature_cols + [
    "impression_change_pct"
]

X_leaky = model_data[leaky_feature_cols]
y_leaky = model_data["is_declining"]

In [60]:
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.25,
    random_state=42,
    stratify=y_leaky
)

In [61]:
leaky_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

leaky_model.fit(
    X_train_leaky,
    y_train_leaky
)

leaky_pred = leaky_model.predict(X_test_leaky)
leaky_prob = leaky_model.predict_proba(X_test_leaky)[:, 1]

In [62]:
print("LEAKY MODEL")
print("=" * 50)

print(
    classification_report(
        y_test_leaky,
        leaky_pred,
        digits=3
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_test_leaky,
            leaky_prob
        ),
        3
    )
)

LEAKY MODEL
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     12238
           1      1.000     1.000     1.000     12986

    accuracy                          1.000     25224
   macro avg      1.000     1.000     1.000     25224
weighted avg      1.000     1.000     1.000     25224

ROC-AUC: 1.0


Performance that is suspiciously high and perfect. Because the model has been given:

```text
impression_change_pct
        ↓
    if < -20%
        ↓
  is_declining = 1
```

The model doesn't need to predict the future.

It already has information calculated using the future.

In [63]:
data[
    [
        "imp_prev30",
        "imp_next30",
        "impression_change_pct",
        "is_declining"
    ]
].head(10)

,imp_prev30,imp_next30,impression_change_pct,is_declining
0,122.0,377.0,2.090164,0
1,148.0,59.0,-0.601351,1
2,2585.0,10112.0,2.911799,0
3,1143.0,1892.0,0.655293,0
4,347.0,167.0,-0.518732,1
5,539.0,162.0,-0.699443,1
6,468.0,197.0,-0.579060,1
7,536.0,1023.0,0.908582,0
8,647.0,475.0,-0.265842,1
9,277.0,95.0,-0.657040,1


## Remove the leakage


In [64]:
data = data.drop(columns=["impression_change_pct"])

X = data[feature_cols]
y = data["is_declining"]

## Final comparison

In [65]:
print("LEAKAGE EXPERIMENT")
print("=" * 60)

print(
    f"Honest model ROC-AUC: "
    f"{roc_auc_score(y_test, honest_prob):.3f}"
)

print(
    f"Leaky model ROC-AUC:  "
    f"{roc_auc_score(y_test_leaky, leaky_prob):.3f}"
)

LEAKAGE EXPERIMENT
Honest model ROC-AUC: 0.731
Leaky model ROC-AUC:  1.000


## Leakage Trap

I deliberately introduced `impression_change_pct` as a model feature.

This feature is calculated using both the previous 30-day impressions and the future 30-day impressions. Because the future 30-day impressions are only known after the prediction period has ended, this feature would not be available at the prediction moment.

The target is defined as:

`is_declining = 1` when future impressions decline by more than 20%.

Therefore, `impression_change_pct` contains direct information about the target definition. The model can use this information to identify the label rather than genuinely predicting future decline.

The leaky model achieved substantially stronger performance than the honest model. This performance is misleading and does not represent real predictive ability.

After demonstrating the leakage, I removed `impression_change_pct` from the feature set.

The final model uses only features that were available before the prediction moment:

- `imp_prev30`
- `clk_prev30`
- `avg_position_prev30`
- `days_with_impressions`
- `content_age_days`

This experiment demonstrates why the feature window and target window must be strictly separated. A model that has access to future information can appear highly accurate while being useless for real-world prediction.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset supports analysis of observed search and engagement behavior, but it has several important limitations.

### 1. Unbalanced history

The panel is unbalanced. Different clients have different amounts of historical data, and client tracking begins at different dates.

Therefore, the absence of earlier observations does not necessarily mean that a page had zero search activity.

### 2. GSC and GA4 availability differ

Some clients have GSC data but do not yet have GA4 data for the same period.

For this lane, GSC availability is particularly important because the target is based on organic search impressions.

Rows where GSC data is unavailable should not automatically be interpreted as zero search performance.

### 3. Future labels require careful window construction

The preferred target is a future observed outcome.

The feature window and target window must not overlap.

Features must be calculated only from information available before the prediction point. Future-window metrics can be used to create the label but cannot be used as model features.

### 4. The starter dataset is not sufficient for a true future-window label

The starter CSV contains aggregated and derived fields but does not provide the full daily history needed to construct arbitrary historical and future windows.

The warehouse daily fact table is therefore preferred for the final future-window prediction task.

### 5. The data cannot establish causality

The dataset can show associations between historical signals and later performance outcomes.

It cannot prove that refreshing a page caused its search performance to recover.

An observed recovery may have multiple explanations, including seasonality, competition, search demand changes, or other external factors.

### 6. The data cannot explain Google's ranking algorithm

The analysis can identify directional relationships between observed page-level signals and measured search performance.

It cannot establish Google's algorithmic factors or prove why Google ranked a particular page higher or lower.

### 7. Query data has a fixed 90-day window

The query-level table must be checked carefully before use.

A query feature is safe only if its observation window ends at or before the prediction moment. If the 90-day window overlaps the future target period, it must be excluded to avoid leakage.

### Named limitation

The main limitation of this initial slice is **uneven historical coverage across clients**. Because clients entered tracking at different times, a March 2026 feature window may not represent the same amount of historical context for every page. This may affect comparability and must be considered when designing the final validation strategy.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.